# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** one row = one content item (`content_hash_id`) belonging to one client
(`client_hash_id`), with its organic search performance aggregated over one full calendar month.

**Table(s):** `fact_content_daily_performance` (daily grain, rolled up to monthly per content
item), joined with `dim_content` (one row per content item, for content metadata).

**Time window:** March 2026 (`month = '2026-03'`) — a mid-panel month, not the sealed final
month (June 2026) and not the `_sample` table, per the panel-iteration rule.

**What I'd predict/rank:** an engagement-opportunity score per content item — how far its
observed organic CTR falls below the CTR expected for its average position that month, weighted
by impression volume. A ranking/scoring proxy, not a strict yes/no label.

**Deliberately excluded:** `fact_content_query_90d`. Its 90-day rolling window doesn't align
cleanly with a single calendar month, and risks the target period leaking into the feature
period. Monthly aggregates from `fact_content_daily_performance` are enough for this lane.

In [9]:

%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# GRAIN CHECK on the source daily table: one row per (report_date, client, content)?
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows violating the daily grain (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the daily grain (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks` (monthly sum) | **Label ingredients** | Used only to compute the CTR target — never used as a feature (see the trap in Part 3) |
| `gsc_avg_position` (monthly avg) | Feature | Known at month-end, not derived from the label |
| `word_count`, `content_type` (dim_content) | Feature | Static content attributes, fixed before the month starts |
| `content_updated_date` → days-since-update | Feature | Historical fact, always before the analysis month |
| `client_hash_id`, `content_hash_id` | Context | Grouping/joining only — never a model feature |
| `gsc_data_available`, `is_published`, `is_deleted` | Context (filter) | Used to filter rows in/out, never as a feature value |
| `sessions_organic/direct/referral/social/paid/ai`, `ai_chatgpt/perplexity/gemini/copilot/claude/meta/other` | Excluded | Out of scope for this lane's organic-search CTR question — mixing channels would blur the target |

In [10]:
con.sql(f"""
    SELECT content_type, COUNT(*) AS n
    FROM {DIM_CONTENT}
    GROUP BY content_type
    ORDER BY n DESC
""").df()


,content_type,n
0,keyword article,459174
1,feedly article,57024
2,comparison article,3408


## 3. Verify it with queries (grain, counts, missing values, windows)

**Fact 1 (grain):** proven in Part 1 above — zero rows violate the daily grain.

**Fact 2 (row count + date span):** see output below.

**Fact 3 (availability, filtered with `IS TRUE`):** see output below.

**Five features (max), each knowable before the decision moment:**
1. `ctr_observed` — knowable because it's fully observed for the completed month being analyzed.
2. `avg_position` — knowable because GSC position is measured concurrently with impressions.
3. `word_count` — knowable because content is already published before the analysis month.
4. `content_type` — knowable because it's a static attribute set at content creation.
5. `days_since_update` — knowable because `content_updated_date` is always a past fact.

**The trap:** `gsc_clicks` is the numerator used to compute `ctr_observed`. Adding it as a
"feature" lets a model trivially reconstruct the target instead of learning anything real —
shown and removed below.

In [11]:
# --- Fact 2: row count + date span of our working slice ---
counts = con.sql(f"""
    SELECT COUNT(*) AS n_daily_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {FACT}
    WHERE month = '2026-03'
""").df()
print("Row count + date span for month=2026-03:")
print(counts)

# --- Fact 3: availability, filtered with IS TRUE (never = TRUE or NOT — NULLs exist) ---
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM {FACT}
    WHERE month = '2026-03'
""").df()
print("\nAvailability check (IS TRUE):")
print(avail)

# --- Build the monthly per-content feature frame (5 features max) ---
features = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
        AVG(f.gsc_avg_position) AS avg_position,
        ANY_VALUE(d.word_count) AS word_count,
        ANY_VALUE(d.content_type) AS content_type,
        DATE '2026-03-31' - ANY_VALUE(d.content_updated_date) AS days_since_update
    FROM {FACT} f
    JOIN {DIM_CONTENT} d
      ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id, f.client_hash_id
""").df()

print(f"\nFeature frame: {len(features)} rows, {features['content_hash_id'].nunique()} unique content items")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row count + date span for month=2026-03:
   n_daily_rows  n_content_items first_date  last_date
0       9841378           331437 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check (IS TRUE):
   total_rows  gsc_available_rows
0     9841378           3611061.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature frame: 176568 rows, 176568 unique content items


,content_hash_id,client_hash_id,ctr_observed,avg_position,word_count,content_type,days_since_update
0,content_04c67f3541177192,client_0797ff3a1fc9a6a5,0.006042,14.129210,3168,keyword article,34
1,content_12890868e4cdac06,client_0797ff3a1fc9a6a5,0.000000,19.000000,3848,keyword article,-50
2,content_14df4b67b008d942,client_0797ff3a1fc9a6a5,0.025641,9.168651,3201,keyword article,-50
3,content_1fea2f270f3c1350,client_0797ff3a1fc9a6a5,0.000000,4.166667,4071,keyword article,34
4,content_20346a450ede60c6,client_0797ff3a1fc9a6a5,0.000000,7.087500,3783,keyword article,34


In [12]:
# --- THE TRAP: add ONE label-derived column on purpose ---
model_data = con.sql(f"""
    SELECT f.content_hash_id,
           SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
           AVG(f.gsc_avg_position) AS avg_position
    FROM {FACT} f
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id
""").df().dropna()

y = model_data['ctr_observed']

# Honest model: only avg_position as a feature
X_honest = model_data[['avg_position']]
honest_r2 = r2_score(y, LinearRegression().fit(X_honest, y).predict(X_honest))
print(f"Honest model R² (avg_position only): {honest_r2:.3f}")

# THE TRAP: sneak in one column that IS the label, just renamed
model_data['leaky_ctr_signal'] = model_data['ctr_observed']
X_leaky = model_data[['avg_position', 'leaky_ctr_signal']]
leaky_r2 = r2_score(y, LinearRegression().fit(X_leaky, y).predict(X_leaky))
print(f"Leaky model R² (+ leaky_ctr_signal): {leaky_r2:.3f}  <- jumps to ~1.0")

print("\n'leaky_ctr_signal' is ctr_observed itself, just renamed.")
print("This is what leakage looks like in its starkest form: a column that secretly IS")
print("the answer lets a model score perfectly by lookup, not by learning any real pattern.")
print("Removing it: only avg_position (and the Part-2 features) stay in the real model.")

Honest model R² (avg_position only): 0.002
Leaky model R² (+ leaky_ctr_signal): 1.000  <- jumps to ~1.0

'leaky_ctr_signal' is ctr_observed itself, just renamed.
This is what leakage looks like in its starkest form: a column that secretly IS
the answer lets a model score perfectly by lookup, not by learning any real pattern.
Removing it: only avg_position (and the Part-2 features) stay in the real model.


## 4. Data limits

**Limitation:** this slice only includes clients with GSC-available content in March 2026 —
46 of 104 total clients. The other 58 clients are entirely absent from this month's contract;
any pattern found here may not generalize to them, especially newly onboarded clients.

**Additional limitation (a real one, not cosmetic):** 148,767 of 176,568 rows (84%) show a
negative "days since update" value — meaning their recorded `content_updated_date` falls
*after* March 2026, not before it. This suggests `content_updated_date` reflects a single
current/latest update timestamp rather than a month-specific history. Practically, this means
a "days since update as of March" feature built this way is **not actually knowable at the
March decision moment** for most rows — it's using information from the future relative to
the analysis window. For real modeling (Week 5+), this field needs a proper historical version
or should be dropped as a feature until one is confirmed available.

In [13]:

total_clients = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/dim_clients.parquet')").fetchone()[0]
clients_in_slice = features['client_hash_id'].nunique()
negative_days = (features['days_since_update'] < 0).sum()

print(f"Total clients in warehouse: {total_clients}")
print(f"Clients represented in this March-2026 slice: {clients_in_slice}")
print(f"Missing from this slice: {total_clients - clients_in_slice} clients")
print(f"\nRows with negative days_since_update (data quirk): {negative_days} of {len(features)}")

Total clients in warehouse: 104
Clients represented in this March-2026 slice: 46
Missing from this slice: 58 clients

Rows with negative days_since_update (data quirk): 148767 of 176568


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.